# 🧠 KET-RAG: Knowledge-Enhanced Triple Retrieval-Augmented Generation
### Medical Q&A over PubMed Diabetes Corpus

**Pipeline Overview:**
1. **Setup** — Clone repo, install dependencies
2. **Preprocessing** — Load data, chunk documents, build FAISS vector store
3. **Skeleton Knowledge Graph** — PageRank-based chunk selection via embedding similarity
4. **Triple Extraction** — LLM-based KG triple generation from top chunks
5. **KG Construction** — Clean, normalize, and build final Knowledge Graph
6. **KG-Augmented Retrieval** — Combine vector + graph retrieval for richer context
7. **GraphRAG Inference** — Answer queries using fused context

---
## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/Neural-GPT/Medical-Rag-Chatbot.git
%cd Medical-Rag-Chatbot
!pip install -r requirements.txt
!pip install rapidfuzz

In [ ]:
import os
import re
import json
import time
import random
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import networkx as nx
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz, process
from groq import Groq

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DataFrameLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from src.preprocessing import load_data, split_docs
from src.vector import retrieve, generate_vector_store

---
## 2. Preprocessing & Vector Store

In [ ]:
DATA_PATH      = "data/pubmed_diabetes_1000_meta.csv"
VECTOR_DB_PATH = "src/Vector_db_850_large"
TRIPLES_FILE   = "chunk_triples.json"
NORM_MAP_FILE  = "entity_norm_map.json"
FINAL_KG_FILE  = "final_kg.json"

# Load and chunk documents
documents = load_data(DATA_PATH, DataFrameLoader)
text = split_docs(
    documents,
    RecursiveCharacterTextSplitter,
    chunk_size=850,
    chunk_overlap=150
)

# Build or load FAISS vector store
generator = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"}
)
vector_store = generate_vector_store(generator, VECTOR_DB_PATH, text, FAISS)

retriever = vector_store.as_retriever(
    search_type="mmr", k=4, fetch_k=10, lambda_mult=0.75
)
chunks_dict = {i: doc.page_content for i, doc in enumerate(vector_store.docstore._dict.values())}
chunks      = list(chunks_dict.values())
print(f"✅ Vector store loaded: {len(chunks)} chunks")

---
## 3. Skeleton Knowledge Graph — PageRank on Embedding Similarity

Build a directed graph where edges connect semantically similar chunks (cosine similarity > threshold). Use PageRank to identify globally important chunks, then deduplicate near-identical ones.

In [ ]:
# ── Build embedding similarity graph ──────────────────────────────────────────
embeddings = np.array(vector_store.index.reconstruct_n(0, vector_store.index.ntotal))
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)  # L2 normalize

K_NEIGHBORS = 100
SIM_THRESHOLD = 0.7

G = nx.DiGraph()
D, I = vector_store.index.search(embeddings, K_NEIGHBORS)

for i in range(len(chunks)):
    for j, score in zip(I[i], D[i]):
        if i != j and score > SIM_THRESHOLD:
            G.add_edge(i, j, weight=float(score))

print(f"Skeleton graph — k: {K_NEIGHBORS}, threshold: {SIM_THRESHOLD}, "
      f"nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")

In [ ]:
# ── PageRank + top-chunk selection ────────────────────────────────────────────
scores       = nx.pagerank(G, weight='weight')
sorted_chunks = sorted(scores.items(), key=lambda x: x[1], reverse=True)

TOP_FRACTION  = 0.33
top_k         = int(TOP_FRACTION * len(sorted_chunks))
top_indices   = [idx for idx, _ in sorted_chunks[:top_k]]
top_embeddings = embeddings[top_indices]

# ── Deduplicate near-identical chunks (cosine > 0.95) ─────────────────────────
DEDUP_THRESHOLD = 0.95
keep_flags = np.ones(len(top_embeddings), dtype=bool)

for i in range(len(top_embeddings)):
    if not keep_flags[i]:
        continue
    sims = np.dot(top_embeddings[i+1:], top_embeddings[i])
    for dup in np.where(sims > DEDUP_THRESHOLD)[0]:
        keep_flags[i + 1 + dup] = False

final_indices = [top_indices[i] for i, flag in enumerate(keep_flags) if flag]
final_chunks  = [chunks[i] for i in final_indices]

print(f"✅ PageRank top-{TOP_FRACTION*100:.0f}%: {len(top_indices)} → {len(final_chunks)} after dedup")

# Preview top 5 chunks
for i in final_indices[:5]:
    print(f"\nScore: {scores[i]:.4f}\n{chunks[i][:300]}\n{'─'*50}")

---
## 4. Triple Extraction via LLM

Extract (head, relation, tail) triples from each important chunk using the Groq LLM. Supports resuming interrupted runs via incremental JSON saves.

In [ ]:
# ── Load Groq client ──────────────────────────────────────────────────────────
with open("/content/GROQ_API_KEY.txt") as f:
    os.environ["GROQ_API_KEY"] = f.read().strip()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL  = "qwen/qwen3-32b"

def llm(prompt: str, system: str = "") -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=1024)
    return resp.choices[0].message.content

In [ ]:
TRIPLETS_PROMPT = """
Extract meaningful knowledge graph triples from the text.

Rules:
- Each triple must be factual and meaningful
- Avoid vague entities like "this", "it", "they"
- Keep entities concise
- Use clear relations (causes, affects, is a type of, etc.)
- Extract at least 1–5 triples if possible

Text:
{chunk}

Return triples in any of these formats:
(head | relation | tail)
head -> relation -> tail
(head, relation, tail)
"""

def parse_triple_line(line: str):
    """Parse a single line into a (head, relation, tail) tuple."""
    if "|" in line:
        parts = line.split("|")
    elif "->" in line:
        parts = line.split("->")
    elif "," in line:
        parts = line.strip("()").split(",")
    else:
        return None
    parts = [p.strip() for p in parts]
    return tuple(parts) if len(parts) == 3 else None


def extract_triples_worker(chunk_idx_chunk: tuple, max_retries: int = 3):
    """LLM-based triple extraction with exponential backoff retry."""
    idx, chunk = chunk_idx_chunk
    prompt = TRIPLETS_PROMPT.format(chunk=chunk)

    for attempt in range(max_retries):
        try:
            output = llm(prompt)
            triples = [t for line in output.strip().split("\n") if (t := parse_triple_line(line))]
            return idx, triples
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e):
                wait = (2 ** attempt) * 2 + random.uniform(0, 2)
                print(f"⏳ Rate limit on chunk {idx}, retrying in {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"❌ Error on chunk {idx}: {e}")
                break
    return idx, []

In [ ]:
# ── Resume-aware parallel triple extraction ───────────────────────────────────
if os.path.exists(TRIPLES_FILE):
    with open(TRIPLES_FILE) as f:
        all_triples = {int(k): v for k, v in json.load(f).items()}
    print(f"🔁 Resuming: {len(all_triples)} chunks already processed")
else:
    all_triples = {}

remaining = [(i, chunk) for i, chunk in enumerate(final_chunks) if i not in all_triples]
print(f"🚀 Chunks remaining: {len(remaining)}")

MAX_WORKERS = 3

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = []
    for item in remaining:
        futures.append(executor.submit(extract_triples_worker, item))
        time.sleep(0.3)  # throttle submission rate

    for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting triples"):
        idx, triples = future.result()
        if triples:
            all_triples[idx] = triples
        if len(all_triples) % 20 == 0:  # incremental save
            with open(TRIPLES_FILE, "w") as f:
                json.dump(all_triples, f, indent=2)

with open(TRIPLES_FILE, "w") as f:
    json.dump(all_triples, f, indent=2)
print(f"✅ Triples extracted for {len(all_triples)} chunks → saved to {TRIPLES_FILE}")

---
## 5. KG Construction — Clean, Normalize & Build Graph

In [ ]:
# ── Load and clean raw triples ─────────────────────────────────────────────────
with open(TRIPLES_FILE) as f:
    raw_data = json.load(f)

GARBAGE_PATTERN = re.compile(r"(maybe|the text|this suggests|\bso\b|i think)", re.IGNORECASE)

def is_clean(triple) -> bool:
    if not isinstance(triple, list) or len(triple) != 3:
        return False
    return all(
        isinstance(p, str) and len(p) <= 120 and not GARBAGE_PATTERN.search(p)
        for p in triple
    )

clean_data = {int(k): [t for t in v if is_clean(t)] for k, v in raw_data.items()}
clean_data = {k: v for k, v in clean_data.items() if v}
print(f"Chunks before cleaning: {len(raw_data)} → after: {len(clean_data)}")

In [ ]:
# ── Flatten triples ────────────────────────────────────────────────────────────
all_raw_triples = [
    (s.strip(), r.strip(), o.strip(), int(cid))
    for cid, triples in clean_data.items()
    for s, r, o in triples
]
print(f"Total raw triples: {len(all_raw_triples)}")

In [ ]:
# ── Entity normalization (fuzzy deduplication) ─────────────────────────────────
def build_norm_map(all_entities: list, threshold: int = 88) -> dict:
    """Fuzzy-group near-duplicate entity names. Returns {variant: canonical}."""
    if os.path.exists(NORM_MAP_FILE):
        with open(NORM_MAP_FILE) as f:
            print("✅ Loaded existing norm map.")
            return json.load(f)

    entities = sorted(set(e.lower().strip() for e in all_entities))
    groups   = []

    for entity in tqdm(entities, desc="Normalizing entities"):
        matched = False
        for group in groups:
            if fuzz.token_sort_ratio(entity, next(iter(group))) >= threshold:
                group.add(entity)
                matched = True
                break
        if not matched:
            groups.append({entity})

    norm_map = {variant: max(group, key=len) for group in groups for variant in group}

    with open(NORM_MAP_FILE, "w") as f:
        json.dump(norm_map, f, indent=2)
    print(f"✅ Normalized {len(entities)} → {len(groups)} unique entities")
    return norm_map


def apply_norm(triple: tuple, norm_map: dict) -> tuple:
    s, r, o, cid = triple
    return (
        norm_map.get(s.lower().strip(), s.lower().strip()),
        r.lower().strip(),
        norm_map.get(o.lower().strip(), o.lower().strip()),
        cid
    )


all_entities   = [s for s, *_ in all_raw_triples] + [o for _, _, o, _ in all_raw_triples]
norm_map       = build_norm_map(all_entities)
normalized_triples = [apply_norm(t, norm_map) for t in all_raw_triples]
print(f"Total normalized triples: {len(normalized_triples)}")

In [ ]:
# ── Build final Knowledge Graph ────────────────────────────────────────────────
KG = nx.DiGraph()

for s, r, o, cid in normalized_triples:
    if KG.has_edge(s, o):
        KG[s][o]["relations"].append(r)
        KG[s][o]["chunk_ids"].append(cid)
    else:
        KG.add_edge(s, o, relations=[r], chunk_ids=[cid])

print(f"✅ KG: {KG.number_of_nodes()} nodes, {KG.number_of_edges()} edges")

# Build inverted index: entity → chunk IDs
entity_to_chunks: dict = defaultdict(set)
for s, r, o, cid in normalized_triples:
    entity_to_chunks[s].add(cid)
    entity_to_chunks[o].add(cid)

# Persist KG
kg_data = [
    {"head": s, "tail": o, "relations": d["relations"], "chunk_ids": d["chunk_ids"]}
    for s, o, d in KG.edges(data=True)
]
with open(FINAL_KG_FILE, "w") as f:
    json.dump(kg_data, f, indent=2)
print(f"✅ KG saved to {FINAL_KG_FILE}")

---
## 6. KG-Augmented Retrieval

Combine dense vector retrieval (semantic similarity) with Knowledge Graph traversal (relational structure) for richer, more grounded context.

In [ ]:
def match_query_to_entities(query: str, norm_map: dict, top_n: int = 5) -> list:
    """Fuzzy-match query tokens to normalized KG entity names."""
    results = process.extract(
        query.lower(),
        list(set(norm_map.values())),
        scorer=fuzz.token_set_ratio,
        limit=top_n
    )
    return [r[0] for r in results if r[1] > 60]


def get_subgraph_chunks(
    matched_entities: list,
    KG: nx.DiGraph,
    entity_to_chunks: dict,
    hops: int = 2,
    max_chunks: int = 6
) -> tuple:
    """BFS up to `hops` from matched entities. Returns (chunk_ids, triples)."""
    visited, frontier  = set(), set(matched_entities) & set(KG.nodes)
    subgraph_triples, chunk_ids = [], set()

    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            if node in visited:
                continue
            visited.add(node)
            for nbr in KG.successors(node):
                edge = KG[node][nbr]
                for r, cid in zip(edge["relations"], edge["chunk_ids"]):
                    subgraph_triples.append((node, r, nbr))
                    chunk_ids.add(cid)
                next_frontier.add(nbr)
            for nbr in KG.predecessors(node):
                edge = KG[nbr][node]
                for r, cid in zip(edge["relations"], edge["chunk_ids"]):
                    subgraph_triples.append((nbr, r, node))
                    chunk_ids.add(cid)
                next_frontier.add(nbr)
        frontier = next_frontier - visited

    return list(chunk_ids)[:max_chunks], subgraph_triples


def kg_augmented_retrieval(
    query: str,
    retriever,
    chunks_dict: dict,
    KG: nx.DiGraph,
    norm_map: dict,
    entity_to_chunks: dict,
    hops: int = 2,
    max_kg_chunks: int = 5
) -> dict:
    """Fuse vector retrieval + KG subgraph traversal into a unified context dict."""
    # Vector retrieval
    vec_texts = [doc.page_content for doc in retriever.invoke(query)]

    # KG retrieval
    matched_entities = match_query_to_entities(query, norm_map)
    kg_chunk_ids, subgraph_triples = get_subgraph_chunks(
        matched_entities, KG, entity_to_chunks, hops=hops, max_chunks=max_kg_chunks
    )
    kg_texts = [chunks_dict[cid] for cid in kg_chunk_ids if cid in chunks_dict]

    # Deduplicate combined context
    seen, combined_chunks = set(), []
    for text in vec_texts + kg_texts:
        key = text[:80]
        if key not in seen:
            seen.add(key)
            combined_chunks.append(text)

    return {
        "chunks":           combined_chunks,
        "subgraph_triples": subgraph_triples,
        "matched_entities": matched_entities,
        "kg_chunk_ids":     kg_chunk_ids,
    }

---
## 7. GraphRAG Inference

In [ ]:
GRAPH_RAG_SYSTEM = (
    "You are a precise medical Q&A assistant. "
    "You will be given:\n"
    "  1. Text chunks from a medical corpus.\n"
    "  2. Knowledge graph triples relevant to the question.\n\n"
    "Rules:\n"
    "  - Answer ONLY from the provided context and triples.\n"
    "  - If the answer is not present, say 'I don\'t know'.\n"
    "  - Be concise. Cite triples when relevant (e.g. [insulin → causes → hypoglycemia])."
)

GRAPH_RAG_TEMPLATE = """
Knowledge Graph Triples:
{triples}

Text Context:
{context}

Question: {question}

Answer:
"""

def format_triples(triples: list, max_triples: int = 30) -> str:
    seen, lines = set(), []
    for s, r, o in triples:
        if (s, r, o) not in seen:
            seen.add((s, r, o))
            lines.append(f"  ({s}) --[{r}]--> ({o})")
        if len(lines) >= max_triples:
            break
    return "\n".join(lines) if lines else "  (no triples found)"


def graph_rag_answer(query: str, context: dict) -> str:
    prompt = GRAPH_RAG_TEMPLATE.format(
        triples=format_triples(context["subgraph_triples"]),
        context="\n\n---\n\n".join(context["chunks"]),
        question=query
    )
    return llm(prompt, system=GRAPH_RAG_SYSTEM)

In [ ]:
# ── Run inference on test queries ──────────────────────────────────────────────
TEST_QUERIES = [
    "What are the causes of diabetes?",
    "How does insulin resistance develop?",
    "What complications arise from Type 2 diabetes?",
    "What is the role of the pancreas in diabetes?",
    "How is diabetes diagnosed?",
]

results = []

for query in tqdm(TEST_QUERIES, desc="Running GraphRAG"):
    context = kg_augmented_retrieval(
        query=query,
        retriever=retriever,
        chunks_dict=chunks_dict,
        KG=KG,
        norm_map=norm_map,
        entity_to_chunks=entity_to_chunks,
    )
    answer = graph_rag_answer(query, context)

    result = {
        "query":            query,
        "answer":           answer,
        "matched_entities": context["matched_entities"],
        "kg_chunks_used":   context["kg_chunk_ids"],
        "total_chunks":     len(context["chunks"]),
        "triples_used":     len(set(map(tuple, context["subgraph_triples"]))),
    }
    results.append(result)

    print(f"\n{'═'*60}")
    print(f"Q: {query}")
    print(f"Entities matched: {context['matched_entities']}")
    print(f"Triples used: {result['triples_used']} | Chunks: {result['total_chunks']}")
    print(f"A: {answer[:500]}...")
    time.sleep(1.5)  # rate limit buffer

with open("graphrag_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n✅ All results saved to graphrag_results.json")